<a href="https://colab.research.google.com/github/alter-mix-dev/PIPELINE/blob/main/Copy_of_Hands_On_Pipeline_Completo_(Soluci%C3%B3n).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PIPELINE COMPLETO: DE DATOS A MODELO DESPLEGADO**

Este Colab conecta en un solo flujo lo visto en los tres Temas anteriores (inferencia básica, RAG y fine-tuning con LoRa) para construir un asistente que recupera contexto propio y responde con un modelo ya ajustado a un tono específico. Es la base directa para el reto del Hackathon 1.

**Nota de alcance:** "Desplegar" aquí significa guardar el modelo ajustado de forma reutilizable y envolverlo en una función lista para usar, no se levanta un servidor ni un endpoint público.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [1]:
# Instalar librerías e iniciar sesión en Hugging Face con el token desde Colab Secrets
#google. prestame a llama
# y dejame tunearla de manera acelerada con retroalimentacion para RAG
!pip install transformers peft accelerate trl sentence-transformers --quiet

import torch
print("¿GPU Disponible?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Modelo de GPU:", torch.cuda.get_device_name(0))
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

login(token=userdata.get('hf_token_jgc'))
print("Sesión de Hugging Face iniciada correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.3 MB/s eta 0:00:00
¿GPU Disponible?: True
Modelo de GPU: Tesla T4
Sesión de Hugging Face iniciada correctamente.


## **PASO 1: LA BASE DE CONOCIMIENTO (RAG)**

Indexamos una base de conocimiento propia con `sentence-transformers` para poder recuperar el fragmento más relevante antes de responder, esto es lo que evita que el asistente alucine sobre políticas que no conoce.

In [2]:
# Generar los embeddings de la base de conocimiento (política de devoluciones)

from sentence_transformers import SentenceTransformer
import numpy as np

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Los libros se prestan por un máximo de 15 días, con posibilidad de renovar una vez si nadie más lo ha solicitado.",
    "Las devoluciones tardías generan una multa de $10 por día de retraso, hasta un "
    "máximo de $150 por libro.",
    "Los libros de la sección de reserva (material de examen) solo se consultan dentro de la biblioteca, no se prestan a domicilio."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)
print(embeddings_documentos.dtype)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)
float32


In [3]:
# Definir la función de recuperación (RAG)

def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

buscar_fragmento("¿Puedo llevarme a casa un libro de la sección de reserva?")

'Los libros se prestan por un máximo de 15 días, con posibilidad de renovar una vez si nadie más lo ha solicitado.'

## **PASO 2: AJUSTAR EL TONO DEL MODELO (FINE-TUNING CON LoRA)**

Cargamos un modelo ligero basado en Llama y lo ajustamos con LoRA para que responda siempre en el mismo tono breve y directo, la configuración de esta celda ya fue validada en el tema 3.

In [4]:
# Cargar el modelo base y su tokenizer

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [5]:
def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=60):
    mensajes = [{"role": "user", "content": pregunta}]
    prompt_formateado = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(prompt_formateado, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id, no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    return tokenizer.decode(tokens_nuevos, skip_special_tokens=True).strip()

prompt_prueba = "¿Puedo renovar un préstamo?"
print(generar_respuesta(modelo, prompt_prueba))

Sí, puedes renovar el préstemo si deseas. La renovación de un préstergo es una operación de reembolso de la cantidad de la inversión original, pero con una nueva fecha de retención. En este caso, debes


In [6]:
# Definir el dataset de ejemplos y convertirlo en Dataset

from datasets import Dataset

def formatear_ejemplo(pregunta, respuesta):
    mensajes = [
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta}
    ]
    return tokenizer.apply_chat_template(mensajes, tokenize=False)

pares = [
    ("¿Puedo renovar un préstamo?", "Sí, puedes renovarlo una vez si nadie más lo ha solicitado."),
    ("¿Hasta cuándo puedo tener un libro prestado?", "El préstamo estándar es de 15 días."),
    ("¿Necesito credencial para entrar a la biblioteca?", "Sí, es necesario mostrar tu credencial vigente en la entrada."),
    ("¿Puedo reservar una sala de estudio?", "Sí, puedes reservarla hasta con 2 días de anticipación desde el portal."),
    ("¿Hay wifi disponible dentro de la biblioteca?", "Sí, la red 'Biblioteca-Invitados' está disponible en todas las salas."),
]

ejemplos = [{"texto": formatear_ejemplo(p, r)} for p, r in pares]
dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

In [7]:
# Probar el modelo base con el prompt de prueba antes de ajustarlo

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

Sí, puedes renovar el préstemo si deseas. La renovación de un préstergo es una operación de reembolso de la cantidad de la inversión original, pero con una nueva fecha de retención. En este caso, debes


In [8]:
# Configurar LoRA (semilla fija para un resultado reproducible en la grabación)

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [9]:
# Entrenar con LoRA

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados_pipeline",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = trainer.state.log_history[-2]['loss'] # último paso del entrenamiento
#perdida_final = resultado_entrenamiento.training_loss # promedio del entrenamiento
print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.808', 'grad_norm': '2.961', 'learning_rate': '0.0002', 'entropy': '1.368', 'num_tokens': '251', 'mean_token_accuracy': '0.5976', 'epoch': '1'}
{'loss': '2.748', 'grad_norm': '2.762', 'learning_rate': '0.0001933', 'entropy': '1.372', 'num_tokens': '502', 'mean_token_accuracy': '0.6016', 'epoch': '2'}
{'loss': '2.657', 'grad_norm': '2.782', 'learning_rate': '0.0001867', 'entropy': '1.377', 'num_tokens': '753', 'mean_token_accuracy': '0.6016', 'epoch': '3'}
{'loss': '2.553', 'grad_norm': '2.799', 'learning_rate': '0.00018', 'entropy': '1.379', 'num_tokens': '1004', 'mean_token_accuracy': '0.6016', 'epoch': '4'}
{'loss': '2.441', 'grad_norm': '2.887', 'learning_rate': '0.0001733', 'entropy': '1.388', 'num_tokens': '1255', 'mean_token_accuracy': '0.6016', 'epoch': '5'}
{'loss': '2.322', 'grad_norm': '3.036', 'learning_rate': '0.0001667', 'entropy': '1.399', 'num_tokens': '1506', 'mean_token_accuracy': '0.6057', 'epoch': '6'}
{'loss': '2.207', 'grad_norm': '3.215', 'learning_rate

## **PASO 3: DESPLEGAR EL MODELO AJUSTADO**

"Desplegar" en este contexto significa guardar el adaptador LoRA de forma reutilizable, no levantar un servidor. Con `save_pretrained` queda listo para volver a cargarlo en cualquier notebook sin repetir el entrenamiento; `push_to_hub` es opcional si quieres tenerlo disponible en tu cuenta de Hugging Face.

In [10]:
# Guardar el adaptador LoRA localmente

modelo_lora.save_pretrained("/content/modelo_ajustado_lora")
print("Adaptador LoRA guardado en /content/modelo_ajustado_lora")

# Opcional: subir el adaptador a tu cuenta de Hugging Face para reutilizarlo fuera de esta sesión
# modelo_lora.push_to_hub("tu-usuario/tinyllama-atencion-clientes-lora")

Adaptador LoRA guardado en /content/modelo_ajustado_lora


## **PASO 4: EL PIPELINE COMPLETO — RAG + MODELO AJUSTADO**

Con la base de conocimiento indexada y el modelo ya ajustado, conectamos ambas piezas en una sola función: recupera el fragmento relevante, se lo entrega al modelo ajustado junto con la pregunta, y genera la respuesta final. Este es el mismo patrón que se espera construir en el Hackathon 1.

***Nota:** TinyLlama aprendió un tono específico con solo 5 ejemplos, suficiente para demostrar que LoRA funciona, pero no para generalizar a una instrucción nueva con contexto inyectado, como la que pide este pipeline con RAG.*

*Por lo que, un modelo real desplegado sí tendría esa capacidad, usando aquí Groq (GPT-OSS-20B) en lugar del modelo ajustado con LoRA como muestra de ese comportamiento.*

In [14]:
# Función del asistente: RAG (recuperación) + modelo desplegado (generación)

!pip install groq -q

from groq import Groq
client = Groq(api_key=userdata.get('api-jgc'))

def asistente(pregunta):
    fragmento = buscar_fragmento(pregunta)
    prompt = f"""Responde la pregunta del alumno usando SOLO el siguiente reglamento de la biblioteca. Si el reglamento no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

for pregunta in [
    "¿Puedo llevarme a casa un libro de la sección de reserva (material de examen)?",
    "¿Cuánto es la multa por entregar un libro tarde?",
    "¿Puedo renovar el préstamo de un libro?",
]:
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {asistente(pregunta)}\n")

Pregunta: ¿Puedo llevarme a casa un libro de la sección de reserva (material de examen)?
Respuesta: No, no se prestan a domicilio.

Pregunta: ¿Cuánto es la multa por entregar un libro tarde?
Respuesta: $10 por día de retraso, con un máximo de $150 por libro.

Pregunta: ¿Puedo renovar el préstamo de un libro?
Respuesta: Sí, una vez si nadie ha solicitado el libro.

